# NB01: Contamination Probe

Test all 8 models for Blood Meridian memorization before benchmarking.

Approach:
- Feed opening ~100 words from 5+ chapters
- Ask models to "reproduce the original text"
- Measure verbatim reproduction: longest common substring, normalized edit distance, BLEU-4
- Flag models with >30% verbatim reproduction

Cost estimate: ~$1

Depends on: NB00 (fingerprint saved)

In [ ]:
import sys
sys.path.insert(0, '..')

import re
import json
import numpy as np
import pandas as pd
from pathlib import Path
from Levenshtein import ratio as lev_ratio
from sacrebleu.metrics import BLEU

from lib.config import (
    CORPUS_PATH, RESULTS_DIR, MODEL_ORDER, MODELS, SEED
)
from lib.api_clients import generate
from lib.prompts import build_contamination_prompt

RESULTS_PATH = RESULTS_DIR / 'contamination'
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

## 1. Extract Probe Passages

Take the first ~100 words from 5 different chapters as prompts.
We also need the "ground truth" continuation (~200 words after the prompt).

In [ ]:
# Load corpus
raw = CORPUS_PATH.read_text(encoding='utf-8')
lines = raw.split('\n')
corpus = '\n'.join(lines[55:])

# Split into chapters (look for chapter markers)
# Blood Meridian chapters typically start with Roman numerals or chapter headings
# We'll use a simpler approach: split into roughly equal segments
words = corpus.split()
total_words = len(words)
print(f'Total words: {total_words:,}')

# Take 5 evenly spaced passages
probe_positions = [0, total_words // 5, 2 * total_words // 5,
                   3 * total_words // 5, 4 * total_words // 5]

PROMPT_WORDS = 100
CONTINUATION_WORDS = 200

probes = []
for i, pos in enumerate(probe_positions):
    prompt_text = ' '.join(words[pos:pos + PROMPT_WORDS])
    continuation_text = ' '.join(words[pos + PROMPT_WORDS:pos + PROMPT_WORDS + CONTINUATION_WORDS])
    probes.append({
        'id': i,
        'position': pos,
        'prompt': prompt_text,
        'ground_truth': continuation_text,
    })
    print(f'\nProbe {i} (word {pos:,}):')
    print(f'  Prompt: {prompt_text[:120]}...')
    print(f'  Truth:  {continuation_text[:120]}...')

## 2. Contamination Metrics

In [ ]:
def longest_common_substring(s1: str, s2: str) -> str:
    """Find the longest common substring between two strings."""
    m, n = len(s1), len(s2)
    # Use word-level for efficiency
    w1 = s1.lower().split()
    w2 = s2.lower().split()
    m, n = len(w1), len(w2)
    longest = 0
    end_pos = 0
    # DP table (only need current and previous row)
    prev = [0] * (n + 1)
    for i in range(1, m + 1):
        curr = [0] * (n + 1)
        for j in range(1, n + 1):
            if w1[i-1] == w2[j-1]:
                curr[j] = prev[j-1] + 1
                if curr[j] > longest:
                    longest = curr[j]
                    end_pos = i
        prev = curr
    lcs_words = w1[end_pos - longest:end_pos]
    return ' '.join(lcs_words)


def contamination_score(generated: str, ground_truth: str) -> dict:
    """Compute contamination metrics between generated text and ground truth."""
    # Normalize
    gen = generated.strip().lower()
    truth = ground_truth.strip().lower()

    # Longest common substring (word-level)
    lcs = longest_common_substring(gen, truth)
    lcs_words = len(lcs.split()) if lcs else 0
    truth_words = len(truth.split())
    lcs_ratio = lcs_words / truth_words if truth_words > 0 else 0

    # Normalized edit distance (Levenshtein)
    edit_sim = lev_ratio(gen, truth)

    # BLEU-4
    bleu = BLEU(effective_order=True)
    bleu_score = bleu.corpus_score([gen], [[truth]]).score

    return {
        'lcs_words': lcs_words,
        'lcs_ratio': lcs_ratio,
        'lcs_text': lcs[:100] if lcs else '',
        'edit_similarity': edit_sim,
        'bleu4': bleu_score,
        'verbatim_flag': lcs_ratio > 0.30,
    }

# Quick test
test = contamination_score('the quick brown fox', 'the quick brown fox jumped over')
print('Test:', test)

## 3. Run Probes Across All Models

In [ ]:
from tqdm.notebook import tqdm
import time

all_results = []

for model_key in tqdm(MODEL_ORDER, desc='Models'):
    label = MODELS[model_key]['label']
    print(f'\n--- {label} ---')

    for probe in tqdm(probes, desc=f'  {label}', leave=False):
        prompt = build_contamination_prompt(probe['prompt'])

        try:
            response = generate(
                model_key, prompt,
                temperature=0.0,  # greedy for max memorization signal
                max_tokens=400,
            )
        except Exception as e:
            print(f'  ERROR on probe {probe["id"]}: {e}')
            response = ''

        scores = contamination_score(response, probe['ground_truth'])
        scores['model'] = model_key
        scores['model_label'] = label
        scores['probe_id'] = probe['id']
        scores['response_preview'] = response[:150]
        all_results.append(scores)

        if scores['verbatim_flag']:
            print(f'  FLAGGED probe {probe["id"]}: LCS={scores["lcs_ratio"]:.1%}')

# Save raw results
results_path = RESULTS_PATH / 'contamination_results.json'
with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'\nSaved to {results_path}')

## 4. Summary Table

In [ ]:
df = pd.DataFrame(all_results)

# Aggregate by model
summary = df.groupby('model_label').agg({
    'lcs_ratio': ['mean', 'max'],
    'edit_similarity': 'mean',
    'bleu4': 'mean',
    'verbatim_flag': 'sum',
}).round(3)

summary.columns = ['Avg LCS Ratio', 'Max LCS Ratio', 'Avg Edit Sim',
                    'Avg BLEU-4', 'Flagged Probes']

# Sort by contamination level
summary = summary.sort_values('Avg LCS Ratio', ascending=False)

print('=== CONTAMINATION SUMMARY ===')
print(summary.to_string())

# Identify contaminated models
contaminated = summary[summary['Flagged Probes'] > 0].index.tolist()
if contaminated:
    print(f'\nContaminated models (flagged): {contaminated}')
    print('These will be marked with * in NB02 results.')
else:
    print('\nNo models flagged for significant memorization.')

# Save summary
summary.to_csv(RESULTS_PATH / 'contamination_summary.csv')

## 5. Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics_to_plot = [
    ('lcs_ratio', 'LCS Ratio (word-level)', 0.30),
    ('edit_similarity', 'Edit Similarity', None),
    ('bleu4', 'BLEU-4', None),
]

for ax, (metric, title, threshold) in zip(axes, metrics_to_plot):
    sns.barplot(data=df, x='model_label', y=metric, ax=ax,
                order=[MODELS[k]['label'] for k in MODEL_ORDER],
                palette='RdYlGn_r')
    ax.set_title(title)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
    if threshold:
        ax.axhline(threshold, color='red', linestyle='--', alpha=0.7, label=f'Threshold={threshold}')
        ax.legend()

plt.tight_layout()
plt.savefig(RESULTS_PATH / 'contamination_plot.png', dpi=150, bbox_inches='tight')
plt.show()